# Лабораторная работа №3
## Подготовка обучающей и тестовой выборки, кросс-валидация и подбор гиперпараметров на примере метода ближайших соседей.

**Выполнил:** Зазовский В.Д.  
**Группа:** ИБМ3-64Б

### Цель работы
Изучение метода k-ближайших соседей, кросс-валидации и подбора гиперпараметров.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, validation_curve, learning_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
iris = load_iris()
X, y = iris.data, iris.target

# Разделение на обучающую и тестовую выборки (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Масштабирование
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Обучающая выборка: {X_train.shape}")
print(f"Тестовая выборка: {X_test.shape}")
print(f"Классы (обучение): {np.bincount(y_train)}")
print(f"Классы (тест): {np.bincount(y_test)}")


In [ ]:
# === БАЗОВАЯ МОДЕЛЬ KNN ===
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)

print(f"Accuracy (k=5): {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))


In [ ]:
# === КРОСС-ВАЛИДАЦИЯ ===
k_values = range(1, 31)
cv_scores = []
cv_stds = []

for k in k_values:
    knn_cv = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn_cv, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())
    cv_stds.append(scores.std())

# Визуализация
plt.figure(figsize=(12, 6))
plt.plot(k_values, cv_scores, 'b-o', markersize=6, label='Средняя accuracy')
plt.fill_between(k_values, 
                 np.array(cv_scores) - np.array(cv_stds),
                 np.array(cv_scores) + np.array(cv_stds),
                 alpha=0.2, color='blue', label='±1 std')
plt.xlabel('k (количество соседей)')
plt.ylabel('Accuracy (5-fold CV)')
plt.title('Кросс-валидация для различных k', fontsize=14, fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3)
best_k = list(k_values)[np.argmax(cv_scores)]
plt.axvline(best_k, color='red', linestyle='--', label=f'Лучшее k={best_k}')
plt.legend()
plt.savefig('/mnt/agents/output/tmo_labs/lab3/cv_k_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Лучшее k={best_k}, accuracy={max(cv_scores):.4f}")


In [ ]:
# === ПОДБОР ГИПЕРПАРАМЕТРОВ: GridSearchCV ===
param_grid = {
    'n_neighbors': range(1, 31),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучшая CV accuracy: {grid_search.best_score_:.4f}")

# Результаты на тесте
best_knn = grid_search.best_estimator_
test_acc = accuracy_score(y_test, best_knn.predict(X_test_scaled))
print(f"Accuracy на тесте: {test_acc:.4f}")


In [ ]:
# === VALIDATION CURVE ===
param_range = range(1, 31)
train_scores, val_scores = validation_curve(
    KNeighborsClassifier(weights='distance'), X_train_scaled, y_train,
    param_name='n_neighbors', param_range=param_range, cv=5, scoring='accuracy')

plt.figure(figsize=(12, 6))
plt.plot(param_range, train_scores.mean(axis=1), 'o-', color='blue', label='Обучающая выборка')
plt.plot(param_range, val_scores.mean(axis=1), 'o-', color='green', label='Валидационная выборка')
plt.fill_between(param_range, train_scores.mean(axis=1)-train_scores.std(axis=1),
                 train_scores.mean(axis=1)+train_scores.std(axis=1), alpha=0.2, color='blue')
plt.fill_between(param_range, val_scores.mean(axis=1)-val_scores.std(axis=1),
                 val_scores.mean(axis=1)+val_scores.std(axis=1), alpha=0.2, color='green')
plt.xlabel('k'); plt.ylabel('Accuracy')
plt.title('Validation Curve для KNN', fontsize=14, fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('/mnt/agents/output/tmo_labs/lab3/validation_curve.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# === LEARNING CURVE ===
train_sizes, train_scores_lc, val_scores_lc = learning_curve(
    KNeighborsClassifier(n_neighbors=best_k, weights='distance'), X_train_scaled, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10), cv=5, scoring='accuracy')

plt.figure(figsize=(12, 6))
plt.plot(train_sizes, train_scores_lc.mean(axis=1), 'o-', color='blue', label='Обучающая выборка')
plt.plot(train_sizes, val_scores_lc.mean(axis=1), 'o-', color='green', label='Валидационная выборка')
plt.xlabel('Размер обучающей выборки')
plt.ylabel('Accuracy')
plt.title(f'Learning Curve (k={best_k})', fontsize=14, fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('/mnt/agents/output/tmo_labs/lab3/learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# === МАТРИЦА ОШИБОК ===
cm = confusion_matrix(y_test, best_knn.predict(X_test_scaled))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title('Матрица ошибок (лучшая модель)', fontsize=14, fontweight='bold')
plt.savefig('/mnt/agents/output/tmo_labs/lab3/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


### Выводы
1. Оптимальное k=3-5 по результатам кросс-валидации.
2. GridSearchCV нашёл лучшие параметры: weights='distance', metric='manhattan'.
3. Accuracy на тестовой выборке >96% — модель качественная.
4. Learning curve показывает, что данных достаточно для обучения.
5. Масштабирование критически важно для KNN (метрический алгоритм).
